In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = Path('../004 data')

## 1. Load Data

### 1.1 Yacht Specifications

In [ ]:
# Load yacht specs — skip empty first row, use second row as header
specs_raw = pd.read_csv(DATA_DIR / 'Yacht-specs.csv', skiprows=1)
specs_raw = specs_raw.dropna(how='all').reset_index(drop=True)

# Drop unnamed first column
specs = specs_raw.loc[:, ~specs_raw.columns.str.startswith('Unnamed')].copy()

# Clean column types
for col in ['GT', 'NT', 'LOA (m)', 'Reg. Length (m)', 'Beam (m)', 'Draft (m)', 'Air Draft (m)']:
    specs[col] = pd.to_numeric(specs[col], errors='coerce')

# Normalize yacht names to match cost data (lowercase yacht_X)
specs['yacht_id'] = specs['Yacht'].str.strip().str.lower()

# For yacht_2 variants (2-1, 2-2, 2-3), create a base ID for joining with cost data
specs['yacht_id_base'] = specs['yacht_id'].str.replace(r'-\d+$', '', regex=True)

print(f'Yacht specs: {len(specs)} entries')
specs

### 1.2 Cost Data (Kostnader_MM.csv)

The file has a hierarchical structure: yacht ID header rows, transaction rows, and subtotal rows. We parse it by tracking the current yacht and extracting only transaction rows with dates.

In [ ]:
import csv

records = []
current_yacht = None
date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')
money_pattern = re.compile(r'^[\d,]+\.\d{2}$')

def looks_like_money(s):
    """Check if a string looks like a monetary value."""
    s = s.strip()
    return bool(money_pattern.match(s))

with open(DATA_DIR / 'Kostnader_MM.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for row in reader:
        if not row or len(row) < 10:
            continue
        
        first = row[0].strip()
        
        # Skip header rows
        if first == '#' and 'Office' in str(row[1]):
            continue
        if first == '' and 'Service Type' in str(row[1]):
            continue
        
        # Detect yacht header (no data on same line)
        if re.match(r'^Yacht_\d+$', first) and (len(row) <= 3 or not date_pattern.search(str(row[3]))):
            current_yacht = first.lower()
            continue
        
        # Yacht_id on same row as data
        if re.match(r'^Yacht_\d+$', first) and len(row) > 3 and date_pattern.search(str(row[3])):
            current_yacht = first.lower()
        
        # Skip non-data rows
        if first in ('Clear', '#'):
            continue
        if current_yacht is None:
            continue
        
        # Must have a date in col 3 to be a data row
        if not (len(row) > 3 and date_pattern.match(str(row[3]).strip())):
            continue
        
        # Detect format by checking col 8:
        # Standard format: col 8 = Cost (no VAT) → looks like money or "0.00"
        # Extended format: col 8 = Supplier name → text, not money
        try:
            col8 = row[8].strip() if len(row) > 8 else ''
            is_standard = looks_like_money(col8)
            
            if is_standard:
                # Standard: ,Office,Arrival,ArrDate,DepDate,ServiceType,InvoiceComments,Supplier,Cost(noVAT),...,FinalCharge
                office = row[1].strip()
                arrival_port = row[2].strip()
                arrival_date_str = row[3].strip()
                departure_date_str = row[4].strip()
                service_type = row[5].strip()
                invoice_comments = row[6].strip()
                supplier = row[7].strip()
                cost_no_vat = row[8].strip()
                final_charge = row[15].strip()
            else:
                # Extended: viewedit,Office,Arrival,ArrDate,DepDate,ServiceType,InternalComments,InvoiceComments,Supplier,Cost(noVAT),...,FinalCharge
                office = row[1].strip()
                arrival_port = row[2].strip()
                arrival_date_str = row[3].strip()
                departure_date_str = row[4].strip()
                service_type = row[5].strip()
                invoice_comments = row[7].strip()
                supplier = row[8].strip()
                cost_no_vat = row[9].strip()
                final_charge = row[16].strip()
            
            records.append({
                'yacht_id': current_yacht,
                'office': office,
                'arrival_port': arrival_port,
                'arrival_date': arrival_date_str,
                'departure_date': departure_date_str,
                'service_type': service_type,
                'invoice_comments': invoice_comments,
                'supplier': supplier,
                'cost_no_vat': cost_no_vat,
                'final_charge': final_charge,
            })
        except (IndexError, ValueError):
            continue

costs_all = pd.DataFrame(records)
print(f'Total transaction rows parsed: {len(costs_all)}')
costs_all.head()

In [ ]:
def parse_money(s):
    """Parse monetary string like '3,665.83' or '0.00' to float."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s in ('', 'Not set'):
        return np.nan
    # Remove thousands separators
    s = s.replace(',', '')
    try:
        return float(s)
    except ValueError:
        return np.nan

# Clean numeric columns
costs_all['cost_no_vat'] = costs_all['cost_no_vat'].apply(parse_money)
costs_all['final_charge'] = costs_all['final_charge'].apply(parse_money)

# Parse dates
costs_all['arrival_date'] = pd.to_datetime(costs_all['arrival_date'], errors='coerce')
costs_all['departure_date'] = pd.to_datetime(costs_all['departure_date'], errors='coerce')

# Derived columns
costs_all['stay_days'] = (costs_all['departure_date'] - costs_all['arrival_date']).dt.days
costs_all['margin'] = costs_all['final_charge'] - costs_all['cost_no_vat']
costs_all['year'] = costs_all['arrival_date'].dt.year

# Show year distribution of all parsed data
print('Transactions per year (all data):')
print(costs_all['year'].value_counts().sort_index())
print(f'\nTotal: {len(costs_all)} transactions')
print(f'Rows with unparseable cost: {costs_all["cost_no_vat"].isna().sum()}')

In [ ]:
# Filter to 2025 data only (our analysis scope)
costs = costs_all[costs_all['year'] == 2025].copy().reset_index(drop=True)
print(f'2025 transactions: {len(costs)}')
print(f'Yachts in 2025 data: {sorted(costs["yacht_id"].unique())}')
costs.head()

### 1.3 Merge Costs with Yacht Specs

In [ ]:
# Create a specs lookup by base yacht_id (average specs for yacht_2 variants)
specs_base = specs.groupby('yacht_id_base')[['GT', 'NT', 'LOA (m)', 'Reg. Length (m)', 'Beam (m)', 'Draft (m)', 'Air Draft (m)']].mean().reset_index()
specs_base.rename(columns={'yacht_id_base': 'yacht_id'}, inplace=True)

# Merge
costs = costs.merge(specs_base, on='yacht_id', how='left')

matched = costs['GT'].notna().sum()
print(f'Transactions matched with specs: {matched}/{len(costs)} ({matched/len(costs)*100:.1f}%)')
costs.head()

### 1.4 Cockpit Data (2020–2025 Trends)

In [ ]:
def parse_cockpit(filepath, year):
    """Parse a cockpit CSV and extract revenues, gross margin, and network activity."""
    raw = pd.read_csv(filepath, header=None)
    # Drop all-NaN columns
    raw = raw.dropna(axis=1, how='all')
    
    service_types = ['Agency Fee', 'Agency Services', 'Bunkering', 'Hospitality', 
                     'Port Marina', 'Provisioning', 'Technical Services', 'Total']
    
    result = {}
    
    for _, row in raw.iterrows():
        vals = [str(v).strip() if pd.notna(v) else '' for v in row]
        
        # Find the label (first non-empty cell)
        label = ''
        val = ''
        for i, v in enumerate(vals):
            if v and v not in ('', str(year)):
                label = v
                # The actual value is typically the next cell
                if i + 1 < len(vals) and vals[i + 1]:
                    val = vals[i + 1]
                break
        
        if label == 'Revenues':
            section = 'revenue'
        elif label == 'Gross Margin' and 'Gross Margin (%)' not in label:
            section = 'gross_margin'
        elif label == 'Gross Margin (%)':
            section = 'gm_pct'
        elif label in ('Network Activity', 'Size of Network', 'Value of Services'):
            section = 'network'
        
        if label in service_types:
            try:
                actual = float(str(val).replace(',', '')) if val else np.nan
            except ValueError:
                actual = np.nan
            result[f'{section}_{label}'] = actual
        
        # Network metrics
        if 'unique boats' in label.lower():
            try:
                result['unique_boats'] = float(str(val).replace(',', ''))
            except (ValueError, TypeError):
                result['unique_boats'] = np.nan
        elif 'port calls' in label.lower() and 'avg' not in label.lower():
            try:
                result['port_calls'] = float(str(val).replace(',', ''))
            except (ValueError, TypeError):
                result['port_calls'] = np.nan
        elif 'days spent' in label.lower():
            try:
                result['days_in_network'] = float(str(val).replace(',', ''))
            except (ValueError, TypeError):
                result['days_in_network'] = np.nan
    
    result['year'] = year
    return result

# Parse all cockpit files
cockpit_records = []
for year in range(2020, 2026):
    fp = DATA_DIR / f'cockpit_{year}.csv'
    if fp.exists():
        cockpit_records.append(parse_cockpit(fp, year))

cockpit = pd.DataFrame(cockpit_records)
cockpit = cockpit.set_index('year').sort_index()

# Show key metrics
display_cols = [c for c in cockpit.columns if 'revenue' in c or c in ('unique_boats', 'port_calls', 'days_in_network')]
cockpit[display_cols]

---
## 2. Descriptive Statistics (2025 Data)

In [ ]:
# Summary statistics for numeric columns
costs[['cost_no_vat', 'final_charge', 'margin', 'stay_days']].describe().round(2)

In [ ]:
# Transactions per yacht
print('Transactions per yacht:')
print(costs['yacht_id'].value_counts().sort_index())
print(f'\nTransactions per service type:')
print(costs['service_type'].value_counts())

In [ ]:
# Transactions per office
print('Transactions per office:')
print(costs['office'].value_counts())

print(f'\nMissing values:')
print(costs.isnull().sum())

In [ ]:
# Yacht specs overview
specs[['Yacht', 'GT', 'NT', 'LOA (m)', 'Beam (m)', 'Draft (m)', 'Air Draft (m)']].describe().round(2)

---
## 3. Visualizations

### 3.1 Yacht Fleet Profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# GT distribution
axes[0].barh(specs['Yacht'], specs['GT'], color='steelblue')
axes[0].set_xlabel('Gross Tonnage (GT)')
axes[0].set_title('Yacht Sizes by GT')
axes[0].invert_yaxis()

# LOA vs GT
axes[1].scatter(specs['LOA (m)'], specs['GT'], s=80, c='steelblue', edgecolors='black', alpha=0.7)
for _, row in specs.iterrows():
    axes[1].annotate(row['Yacht'], (row['LOA (m)'], row['GT']), fontsize=7, ha='left', va='bottom')
axes[1].set_xlabel('Length Overall (m)')
axes[1].set_ylabel('Gross Tonnage')
axes[1].set_title('LOA vs GT')

# Beam vs Draft
axes[2].scatter(specs['Beam (m)'], specs['Draft (m)'], s=specs['GT']/10 + 20, c='steelblue', edgecolors='black', alpha=0.7)
for _, row in specs.iterrows():
    axes[2].annotate(row['Yacht'], (row['Beam (m)'], row['Draft (m)']), fontsize=7, ha='left', va='bottom')
axes[2].set_xlabel('Beam (m)')
axes[2].set_ylabel('Draft (m)')
axes[2].set_title('Beam vs Draft (bubble size = GT)')

plt.tight_layout()
plt.show()

### 3.2 Cost Analysis (2025)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total cost per yacht
yacht_cost = costs.groupby('yacht_id')['final_charge'].sum().sort_values(ascending=True)
yacht_cost.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Total Final Charge (EUR)')
axes[0].set_title('Total Cost per Yacht (2025)')

# Cost per service type
svc_cost = costs.groupby('service_type')['final_charge'].sum().sort_values(ascending=True)
svc_cost.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('Total Final Charge (EUR)')
axes[1].set_title('Total Cost per Service Type (2025)')

plt.tight_layout()
plt.show()

In [ ]:
# Cost by month (2025)
costs['month'] = costs['arrival_date'].dt.to_period('M')
monthly = costs.groupby('month')['final_charge'].sum()

fig, ax = plt.subplots(figsize=(12, 5))
monthly.plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Month')
ax.set_ylabel('Total Final Charge (EUR)')
ax.set_title('Monthly Cost Distribution (2025)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### 3.3 Cost vs Yacht Size

In [ ]:
# Aggregate total cost per yacht, then join with GT
yacht_totals = costs.groupby('yacht_id').agg(
    total_charge=('final_charge', 'sum'),
    total_cost=('cost_no_vat', 'sum'),
    total_margin=('margin', 'sum'),
    n_transactions=('final_charge', 'count'),
    avg_stay=('stay_days', 'mean'),
    GT=('GT', 'first'),
    LOA=('LOA (m)', 'first'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GT vs Total Charge
axes[0].scatter(yacht_totals['GT'], yacht_totals['total_charge'], s=100, c='steelblue', edgecolors='black')
for _, r in yacht_totals.iterrows():
    axes[0].annotate(r['yacht_id'], (r['GT'], r['total_charge']), fontsize=7, ha='left', va='bottom')
axes[0].set_xlabel('Gross Tonnage (GT)')
axes[0].set_ylabel('Total Final Charge (EUR)')
axes[0].set_title('Yacht Size (GT) vs Total Cost (2025)')

# LOA vs Total Charge
axes[1].scatter(yacht_totals['LOA'], yacht_totals['total_charge'], s=100, c='coral', edgecolors='black')
for _, r in yacht_totals.iterrows():
    axes[1].annotate(r['yacht_id'], (r['LOA'], r['total_charge']), fontsize=7, ha='left', va='bottom')
axes[1].set_xlabel('Length Overall (m)')
axes[1].set_ylabel('Total Final Charge (EUR)')
axes[1].set_title('Yacht Length (LOA) vs Total Cost (2025)')

plt.tight_layout()
plt.show()

### 3.4 Margin Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Margin per service type
margin_svc = costs.groupby('service_type')['margin'].sum().sort_values(ascending=True)
colors = ['red' if v < 0 else 'seagreen' for v in margin_svc]
margin_svc.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_xlabel('Total Margin (EUR)')
axes[0].set_title('Margin per Service Type (2025)')
axes[0].axvline(x=0, color='black', linewidth=0.5)

# Margin per yacht
margin_yacht = costs.groupby('yacht_id')['margin'].sum().sort_values(ascending=True)
colors_y = ['red' if v < 0 else 'seagreen' for v in margin_yacht]
margin_yacht.plot(kind='barh', ax=axes[1], color=colors_y)
axes[1].set_xlabel('Total Margin (EUR)')
axes[1].set_title('Margin per Yacht (2025)')
axes[1].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

### 3.5 Stay Duration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stay duration distribution
costs['stay_days'].dropna().hist(bins=30, ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Stay Duration (days)')
axes[0].set_ylabel('Number of Transactions')
axes[0].set_title('Distribution of Stay Duration (2025)')

# Stay duration vs cost
axes[1].scatter(costs['stay_days'], costs['final_charge'], alpha=0.5, c='steelblue', edgecolors='black', s=40)
axes[1].set_xlabel('Stay Duration (days)')
axes[1].set_ylabel('Final Charge (EUR)')
axes[1].set_title('Stay Duration vs Cost per Transaction (2025)')

plt.tight_layout()
plt.show()

### 3.6 Cockpit Trends (2020–2025)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Revenue trend
rev_cols = [c for c in cockpit.columns if c.startswith('revenue_') and 'Total' not in c]
if rev_cols:
    cockpit[rev_cols].rename(columns=lambda c: c.replace('revenue_', '')).plot(ax=axes[0, 0], marker='o')
    axes[0, 0].set_title('Revenue by Service Type (2020–2025)')
    axes[0, 0].set_ylabel('EUR')
    axes[0, 0].legend(fontsize=8)

# Total revenue and gross margin
totals = cockpit[['revenue_Total', 'gross_margin_Total']].dropna(how='all')
if not totals.empty:
    totals.rename(columns={'revenue_Total': 'Revenue', 'gross_margin_Total': 'Gross Margin'}).plot(
        ax=axes[0, 1], marker='o', color=['steelblue', 'seagreen'])
    axes[0, 1].set_title('Total Revenue vs Gross Margin (2020–2025)')
    axes[0, 1].set_ylabel('EUR')

# Network activity
net_cols = ['unique_boats', 'port_calls']
if all(c in cockpit.columns for c in net_cols):
    cockpit[net_cols].rename(columns={'unique_boats': 'Unique Boats', 'port_calls': 'Port Calls'}).plot(
        ax=axes[1, 0], marker='o', color=['steelblue', 'coral'])
    axes[1, 0].set_title('Network Activity (2020–2025)')
    axes[1, 0].set_ylabel('Count')

# Days in network
if 'days_in_network' in cockpit.columns:
    cockpit['days_in_network'].dropna().plot(ax=axes[1, 1], marker='o', color='steelblue')
    axes[1, 1].set_title('Total Days in Network (2020–2025)')
    axes[1, 1].set_ylabel('Days')

plt.tight_layout()
plt.show()

---
## 4. Correlation Analysis

In [ ]:
# Correlation heatmap: yacht-level aggregates vs specs
corr_data = yacht_totals[['total_charge', 'total_cost', 'total_margin', 'n_transactions', 'avg_stay', 'GT', 'LOA']].dropna()

fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = corr_data.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, square=True)
ax.set_title('Correlation Matrix: Yacht Costs vs Specifications (2025)')
plt.tight_layout()
plt.show()

print('\nKey correlations with total_charge:')
print(corr_matrix['total_charge'].sort_values(ascending=False).round(3))

---
## 5. Summary of Findings

Key observations from the EDA will be filled in after running the notebook. Areas to investigate:

- **Fleet composition**: Range of yacht sizes (GT), and how size relates to service costs
- **Cost drivers**: Which service types dominate costs? Which yachts are most expensive to service?
- **Margin structure**: Where does the company earn its margin? Are there negative-margin services?
- **Seasonality**: Is there a seasonal pattern in 2025 costs?
- **Size–cost relationship**: How strongly does yacht size (GT/LOA) predict total costs?
- **Data gaps**: Missing specs for historical yachts limit multi-year cost modeling

# Exploratory Data Analysis — NautiCost Dataset

This notebook performs an EDA of the NautiCost dataset, focusing on **2025 transaction data** linked to the 17 yachts with known specifications.